This notebook follows a deliberate 2-stage curriculum designed to reach ~75% top-1 accuracy on CIFAR-100 from scratch (no pretrained weights).

| Stage | Model | Key technique |
|-------|-------|---------------|
| 1 | ResNet-18 + augmentation + scheduler  | Baseline sanity check |
| 2 | ResNet-50 + CutMix + label smoothing | Stronger regularisation |

**Hardware:** NVIDIA T4 (Kaggle/Colab)  
**Framework:** PyTorch 2.x  
**Dataset:** CIFAR-100 — 50,000 train / 10,000 test, 100 classes, 32×32 RGB

## 0 — Imports and device

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

import numpy as np
import time
import os
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU   :', torch.cuda.get_device_name(0))
    print('VRAM  :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

Device: cuda
GPU   : Tesla T4
VRAM  : 15.6 GB


## 1 — Data loading

Three transform pipelines are defined here and reused across all stages.

- `transform_basic` — the bare minimum (used nowhere in this notebook, shown for reference)
- `transform_train` — what every stage uses for training: normalization + random crop + flip + RandAugment
- `transform_test` — deterministic: normalize only

The CIFAR-100 channel means and stds are dataset-specific constants, not ImageNet values.

In [ ]:
# CIFAR-100 channel statistics (computed over training set)
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

# Training transform
# RandomCrop with padding=4: virtually enlarges each image before cropping,
# giving the model slightly shifted views of each sample.
# RandAugment: randomly applies 2 augmentation operations (rotation, shear,
# colour jitter, posterize, etc.) at magnitude 9 — the standard ViT/ResNet
# recipe for CIFAR-100.
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# Test transform
# No augmentation at test time — deterministic evaluation.
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# Datasets
trainset = torchvision.datasets.CIFAR100(
    root='./data', train=True,  download=True, transform=transform_train
)
testset = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=transform_test
)

# Loaders
# num_workers=2 works on Kaggle; increase to 4 on a local machine.
# pin_memory=True speeds up CPU->GPU transfer on CUDA devices.
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True,
    num_workers=2, pin_memory=True
)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=256, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f'Train batches : {len(trainloader)}  ({len(trainset)} samples)')
print(f'Test  batches : {len(testloader)}  ({len(testset)} samples)')

  0%|          | 0.00/169M [00:00<?, ?B/s]

  0%|          | 65.5k/169M [00:00<04:46, 589kB/s]

  0%|          | 229k/169M [00:00<02:29, 1.13MB/s]

  1%|          | 918k/169M [00:00<00:48, 3.47MB/s]

  2%|▏         | 2.65M/169M [00:00<00:20, 8.30MB/s]

...

 95%|█████████▌| 161M/169M [00:04<00:00, 36.6MB/s]

 97%|█████████▋| 164M/169M [00:04<00:00, 35.9MB/s]

 99%|█████████▉| 168M/169M [00:04<00:00, 36.1MB/s]

100%|██████████| 169M/169M [00:04<00:00, 35.4MB/s]

Train batches : 391  (50000 samples)
Test  batches : 40  (10000 samples)


## 2 — Shared utilities

These three functions are reused across all three stages so the training loop is identical for every model.

In [ ]:
#  CutMix
# CutMix pastes a rectangular crop from one image into another, mixing their
# labels proportionally to the area of the patch.
# This forces the model to classify from partial evidence and is one of the
# single biggest regularisation gains on CIFAR-100.
def cutmix_data(images, labels, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = images.shape
    idx = torch.randperm(B).to(images.device)

    # sample a bounding box
    cut_ratio = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_ratio)
    cut_h = int(H * cut_ratio)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    mixed = images.clone()
    mixed[:, :, y1:y2, x1:x2] = images[idx, :, y1:y2, x1:x2]

    # recompute lambda from actual box area
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed, labels, labels[idx], lam


def cutmix_loss(criterion, outputs, labels_a, labels_b, lam):
    return lam * criterion(outputs, labels_a) + (1 - lam) * criterion(outputs, labels_b)

In [ ]:
# Universal train / evaluate functions
# use_cutmix is toggled per-stage (off for stage 1, on for stages 2 and 3)

def train_one_epoch(model, optimizer, scheduler, criterion, use_cutmix=False):
    model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        if use_cutmix and np.random.rand() > 0.5:
            images, la, lb, lam = cutmix_data(images, labels)
            optimizer.zero_grad()
            outputs = model(images)
            loss = cutmix_loss(criterion, outputs, la, lb, lam)
        else:
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)

        loss.backward()
        # gradient clipping — prevents occasional exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()   # OneCycleLR steps every batch, not every epoch

        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return total_loss / len(trainloader), 100.0 * correct / total


@torch.no_grad()
def evaluate(model):
    model.eval()
    correct = 0
    total   = 0
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return 100.0 * correct / total


def run_training(model, optimizer, scheduler, criterion,
                 epochs, save_path, use_cutmix=False, patience=15):
    best_acc = 0.0
    patience_counter = 0

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(
            model, optimizer, scheduler, criterion, use_cutmix
        )
        test_acc = evaluate(model)
        elapsed  = time.time() - t0

        print(f'Epoch {epoch:3d}/{epochs} | '
              f'loss {train_loss:.4f} | '
              f'train {train_acc:.2f}% | '
              f'test {test_acc:.2f}% | '
              f'{elapsed:.0f}s')

        if test_acc > best_acc:
            best_acc = test_acc
            patience_counter = 0
            torch.save(model.state_dict(), save_path)
            print(f'  --> saved  (best {best_acc:.2f}%)')
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f'Early stop at epoch {epoch}. Best: {best_acc:.2f}%')
            break

    print(f'\nFinal best test accuracy: {best_acc:.2f}%')
    return best_acc

## Stage 1 — ResNet-18 from scratch

1. Adding proper normalization and augmentation to the data pipeline
2. Using a cosine learning rate schedule with linear warmup (OneCycleLR)

Everything else is kept simple. No CutMix yet — evaluating how much augmentation + scheduler alone.

**ResNet-18 struggles on CIFAR-100 (when training from scratch):** it has 11M parameters but its first 7×7 conv with stride 2 immediately halves the 32×32 image to 16×16, losing a lot of spatial detail. This is designed for 224×224 images (we are trying to take care of this problem here).

In [ ]:
# ResNet-18 adjusted for 32×32 input
# The standard ResNet-18 was designed for ImageNet (224×224).
# Its first layer uses kernel=7, stride=2 and is followed by MaxPool(3,2),
# reducing 224->56 (4× downsample). On a 32×32 image this gives 8×8 after
# just the stem — too aggressive.
# Fix: replace the stem with kernel=3, stride=1, no max-pool.
# This is the standard CIFAR adaptation used in the original ResNet paper.

model_s1 = models.resnet18(weights=None)
model_s1.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model_s1.maxpool = nn.Identity()          # remove the maxpool
model_s1.fc = nn.Linear(512, 100)         # 100 CIFAR classes
model_s1 = model_s1.to(device)

total_params = sum(p.numel() for p in model_s1.parameters())
print(f'ResNet-18 (CIFAR stem) — {total_params/1e6:.2f}M parameters')

ResNet-18 (CIFAR stem) — 11.22M parameters


In [ ]:
EPOCHS_S1 = 100

criterion_s1 = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer_s1 = optim.SGD(
    model_s1.parameters(),
    lr=0.1,                 # OneCycleLR will override this
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True           # Nesterov momentum converges slightly faster
)

# OneCycleLR: linearly warms up lr over 30% of training,
# then cosine-decays to near zero.
# This single scheduler is responsible for a large chunk of the accuracy gain
# over a flat learning rate.
scheduler_s1 = optim.lr_scheduler.OneCycleLR(
    optimizer_s1,
    max_lr=0.1,
    epochs=EPOCHS_S1,
    steps_per_epoch=len(trainloader),
    pct_start=0.3,
    anneal_strategy='cos',
    div_factor=25,           # initial lr = max_lr / 25
    final_div_factor=1e4     # final  lr = max_lr / 1e4
)

best_s1 = run_training(
    model_s1, optimizer_s1, scheduler_s1, criterion_s1,
    epochs=EPOCHS_S1,
    save_path='best_stage1_resnet18.pth',
    use_cutmix=False,
    patience=20
)

Epoch   1/100 | loss 4.3874 | train 5.13% | test 10.87% | 42s
  --> saved  (best 10.87%)


Epoch   2/100 | loss 4.0659 | train 9.97% | test 14.30% | 42s
  --> saved  (best 14.30%)


Epoch   3/100 | loss 3.8936 | train 13.10% | test 16.62% | 46s
  --> saved  (best 16.62%)


Epoch   4/100 | loss 3.7304 | train 16.46% | test 22.94% | 50s
  --> saved  (best 22.94%)


Epoch   5/100 | loss 3.5452 | train 20.06% | test 23.89% | 49s
  --> saved  (best 23.89%)


...

Epoch  96/100 | loss 0.9791 | train 95.35% | test 78.27% | 49s
  --> saved  (best 78.27%)


Epoch  97/100 | loss 0.9707 | train 95.67% | test 78.24% | 49s


Epoch  98/100 | loss 0.9621 | train 95.97% | test 78.41% | 49s
  --> saved  (best 78.41%)


Epoch  99/100 | loss 0.9606 | train 96.00% | test 78.28% | 49s


Epoch 100/100 | loss 0.9571 | train 96.15% | test 78.22% | 49s

Final best test accuracy: 78.41%


## Stage 2 — ResNet-50 + CutMix + label smoothing

Two changes from Stage 1:
1. **Larger backbone** — ResNet-50 (25M params vs 11M) with the same CIFAR stem fix
2. **CutMix** — applied randomly to 50% of batches. This is the single most impactful regularisation technique for CIFAR-100 and prevents the model from over-relying on small discriminative regions

Label smoothing (0.1) is carried over from Stage 1. It prevents overconfidence by replacing hard 0/1 targets with 0.001/0.901, which reduces overfitting on a 100-class problem.

In [ ]:
# ResNet-50 adjusted for 32×32 input
model_s2 = models.resnet50(weights=None)
model_s2.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model_s2.maxpool = nn.Identity()
model_s2.fc      = nn.Linear(2048, 100)
model_s2 = model_s2.to(device)

total_params = sum(p.numel() for p in model_s2.parameters())
print(f'ResNet-50 (CIFAR stem) — {total_params/1e6:.2f}M parameters')

ResNet-50 (CIFAR stem) — 23.71M parameters


In [ ]:
EPOCHS_S2 = 120

criterion_s2 = nn.CrossEntropyLoss(label_smoothing=0.1)

# SGD + momentum is the standard for ResNets on CIFAR.
# Weight decay slightly higher than Stage 1 to counter the larger model.
optimizer_s2 = optim.SGD(
    model_s2.parameters(),
    lr=0.1,
    momentum=0.9,
    weight_decay=1e-3,
    nesterov=True
)

scheduler_s2 = optim.lr_scheduler.OneCycleLR(
    optimizer_s2,
    max_lr=0.1,
    epochs=EPOCHS_S2,
    steps_per_epoch=len(trainloader),
    pct_start=0.3,
    anneal_strategy='cos',
    div_factor=25,
    final_div_factor=1e4
)

best_s2 = run_training(
    model_s2, optimizer_s2, scheduler_s2, criterion_s2,
    epochs=EPOCHS_S2,
    save_path='best_stage2_resnet50.pth',
    use_cutmix=True,   # CutMix enabled
    patience=20
)

Epoch   1/120 | loss 4.6172 | train 1.31% | test 1.63% | 195s
  --> saved  (best 1.63%)


Epoch   2/120 | loss 4.5979 | train 1.47% | test 1.86% | 193s


  --> saved  (best 1.86%)


Epoch   3/120 | loss 4.5802 | train 1.85% | test 2.74% | 194s


  --> saved  (best 2.74%)


...

  --> saved  (best 81.01%)


Epoch 117/120 | loss 1.8168 | train 74.94% | test 81.10% | 193s
  --> saved  (best 81.10%)


Epoch 118/120 | loss 1.8496 | train 74.26% | test 81.32% | 193s
  --> saved  (best 81.32%)


Epoch 119/120 | loss 1.8238 | train 75.01% | test 81.49% | 193s
  --> saved  (best 81.49%)


Epoch 120/120 | loss 1.8136 | train 76.33% | test 81.46% | 193s

Final best test accuracy: 81.49%


## 3 Results Summary

| Model | Params | Train acc | Test acc | Gap | Status |
|-------|--------|-----------|----------|-----|--------|
| ResNet-18 | 11M | 96.15% | 78.41% | +17.74% | Capacity-limited |
| ResNet-50 + CutMix | 25M | 76.33% | 81.49% | -5.16% | Still converging |

**Key takeaway:** ResNet-50 with CutMix is the clear winner and the model to carry forward.
The inverted gap on ResNet-50 confirms CutMix is working as intended. ResNet-18 is retired —
its overfitting gap is structural and cannot be resolved without architectural changes.
